In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Import thư viện cần thiết**

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

**Load Data**

In [ ]:
train = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

In [ ]:
train.head(10)
test.head(10)

**Tách dữ liệu x/y**

In [ ]:
y = train["label"].values
X = train.drop(columns=["label"]).values

# Reshape into 28x28 images for DL
X_img = X.reshape(-1, 28, 28, 1)

print("X shape:", X.shape)
print("X_img shape:", X_img.shape)

**Visualization**

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(X_img[i], cmap="gray")
    plt.title(f"Label: {y[i]}")
    plt.axis("off")
plt.suptitle("Examples from the dataset")
plt.show()

**Distribution Plot**

In [ ]:
sns.countplot(x=y)
plt.title("Distribution of digits")
plt.show()

**Normalization
Scale pixel từ 0-255 về 0-1 để mô hình học tốt hơn (CNN yêu cầu input chuẩn hóa)**

In [ ]:
X_norm = (X / 255.0).astype("float32")
X_img_norm = (X_img / 255.0).astype("float32")

print(X_norm.min(), X_norm.max())

**Split train/val ở đây chia 90% train, 10% val
Tăng cường dữ liệu**

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_norm, y, test_size=0.1, stratify=y, random_state=42
)
X_img_norm = X_norm.reshape(-1, 28, 28, 1)
X_train_img, X_val_img, _, _ = train_test_split(
    X_img_norm, y, test_size=0.1, stratify=y, random_state=42
)

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)
datagen.fit(X_train_img)
print("Train ML:", X_train.shape)
print("Train DL:", X_train_img.shape)
print("Val ML:", X_val.shape)
print("Val DL:", X_val_img.shape)

**Xây dựng mô hình CNN
Thiết kế cấu trức mạng gồm các lớp Convolution, Pooling, Fatten và Dense để mô hình học và dự đoán số**

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    
    Flatten(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.summary()

**Compile mô hình**

In [ ]:
model.compile(
    optimizer = Adam(learning_rate=0.001), #thuật toán tối ưu
    loss = 'sparse_categorical_crossentropy', #hàm mất mát cho bài toán nhị phân
    metrics = ['accuracy']#chỉ số đánh giá
)

**Thiết lập callback để tối ưu hóa mô hình huấn luyện**

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

**Huấn luyện mô hình sử dụng datagen**

In [ ]:
batch_size=64
epochs=20

history = model.fit(
    datagen.flow(X_train_img, y_train, batch_size=batch_size),
    validation_data=(X_val_img, y_val),
    epochs=epochs,
    callbacks=[early_stop],
    verbose=1
)

**Đánh giá mô hình**

In [ ]:
val_loss, val_acc = model.evaluate(X_val_img, y_val, verbose=0)
print(f"validation accuracy: {val_acc:.4f}")

**Submission**

In [ ]:
# Chuẩn hóa dữ liệu test
X_test_norm = (test.values / 255.0).astype("float32")
X_test_img = X_test_norm.reshape(-1, 28, 28, 1)  # Keras dùng (N, 28, 28, 1)

# Dự đoán
preds = model.predict(X_test_img)
preds = np.argmax(preds, axis=1)

# Tạo file submission
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(preds) + 1),
    "Label": preds
})

output_path = "/kaggle/working"
submission.to_csv(os.path.join(output_path, "submission_cnn.csv"), index=False)
print("✅ Submission file generated:", os.path.join(output_path, "submission_cnn.csv"))